# **Manual trading challenge: “Invest & Expand”**

You are expanding your outpost into a true market making firm with a budget of `50 000` XIRECs. You need to allocate this budget across three pillars:

- **Research**
- **Scale**
- **Speed**

You choose percentages for each pillar between 0–100%. Total allocation cannot exceed 100%. Your final PnL (Profit and Loss) score is:

<aside>
ℹ️

PnL = (Research × Scale × Speed) − Budget_Used

</aside>

### **The pillars**

**Research** determines how strong your trading edge is. It grows **logarithmically** from `0` (for `0` invested) to `200 000`  (for `100` invested). The exact formula is `research(x) = 200_000 * np.log(1 + x) / np.log(1 + 100)`. Here, `np.log` is a python function from NumPy package for natural logarithm.

**Scale** determines how broadly you deploy your strategy across markets. It grows **linearly** from `0` (for `0` invested) to `7` (for `100` invested).

**Speed** determines how often you win the trades you target. It is **rank-based** across all players:

- Highest speed investment receives a `0.9` multiplier.
- Lowest receives `0.1`.
- Everyone in between is scaled linearly by rank, equal investments share the same rank.
- For example, if people invested `70, 70, 70, 50, 40, 40, 30`, they get the following ranks: `1, 1, 1, 4, 5, 5, 7`. First three players get `0.9` for hit rate multiplier, last player gets `0.1`, and everybody in between gets linearly scaled between top and bottom rank. Another example, if you have three players investing `95, 20, 10`, their ranks are `1, 2, 3`, and their hit rates are `0.9, 0.5, 0.1`.

Your Research, Scale, and Speed outcomes are multiplied together to form your gross PnL, after which the used part of your budget is deducted.

Every decision you make reflects a real trade-off faced by modern market makers: capital is finite, competition is relentless, and edge alone is never enough. Good luck!

## Upgraded market model

This version stops treating speed `c` as an exogenous prior. Instead, each synthetic opponent chooses a full `(a, b, c)` allocation and the notebook projects those choices onto the induced speed distribution.

Latent opponent types:

- **Random/noisy entrants** use the full budget but split it stochastically across the three pillars.
- **Heuristic entrants** choose from rounded anchor allocations like `10/50/40`, `15/45/40`, `20/40/40`, and `20/50/30`.
- **Optimizing entrants** cluster around the best response to the non-strategic crowd.
- **Copycats** imitate the rounded optimizer focal point and nearby `5`-point variants.

Because speed ranking depends only on `c`, the model simulates full allocations first, then collapses the crowd to an induced distribution over `c` for the exact rank-based optimizer.

In [17]:
from dataclasses import asdict, replace
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "datasets" / "round2" / "speed_optimizer.py").exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root / "datasets" / "round2"))

from joint_allocation_model import (
    JointPopulationConfig,
    build_joint_population_model,
    probability_at,
    top_speed_levels,
)


In [18]:
baseline_config = JointPopulationConfig()
baseline_model = build_joint_population_model(baseline_config)

{
    "target_type_shares": baseline_config.type_shares(),
    "realized_type_shares": baseline_model.realized_type_shares(),
    "fixed_point_iterations": baseline_model.fixed_point_iterations,
    "nonstrategic_best_response": asdict(baseline_model.nonstrategic_best_response),
    "optimizer_center": baseline_model.optimizer_center,
    "copycat_center": baseline_model.copycat_center,
    "population_best_response": asdict(baseline_model.population_best_response),
}


{'target_type_shares': {'random': 0.5,
  'heuristic': 0.325,
  'optimizer': 0.1,
  'copycat': 0.075},
 'realized_type_shares': {'random': 0.5,
  'heuristic': 0.325,
  'optimizer': 0.1,
  'copycat': 0.075},
 'fixed_point_iterations': 1,
 'nonstrategic_best_response': {'a': 15.0,
  'b': 45.0,
  'c': 40.0,
  'expected_speed': 0.631872,
  'expected_pnl': 189150.92575910574},
 'optimizer_center': (15, 45, 40),
 'copycat_center': (15, 45, 40),
 'population_best_response': {'a': 15.0,
  'b': 45.0,
  'c': 40.0,
  'expected_speed': 0.6555759999999999,
  'expected_pnl': 198122.41609922814}}

In [19]:
{
    "key_speed_masses": {
        f"p(c={c})": probability_at(baseline_model.speed_distribution, c)
        for c in [30, 35, 40, 45, 50]
    },
    "top_speed_levels": top_speed_levels(
        baseline_model.speed_distribution,
        top_n=10,
    ),
}


{'key_speed_masses': {'p(c=30)': 0.072425,
  'p(c=35)': 0.058735,
  'p(c=40)': 0.259465,
  'p(c=45)': 0.04317,
  'p(c=50)': 0.070545},
 'top_speed_levels': [(40, 0.259465),
  (30, 0.072425),
  (50, 0.070545),
  (35, 0.058735),
  (45, 0.04317),
  (38, 0.022175),
  (42, 0.021905),
  (19, 0.008765),
  (20, 0.00856),
  (25, 0.00856)]}

In [20]:
higher_imitation_config = replace(
    baseline_config,
    random_share=0.47,
    heuristic_share=0.305,
    optimizer_share=0.10,
    copycat_share=0.125,
    seed=11,
)
more_diffuse_config = replace(
    baseline_config,
    random_share=0.55,
    heuristic_share=0.295,
    optimizer_share=0.08,
    copycat_share=0.075,
    seed=13,
)

higher_imitation_model = build_joint_population_model(higher_imitation_config)
more_diffuse_model = build_joint_population_model(more_diffuse_config)

{
    "higher_imitation": {
        "best_response": asdict(higher_imitation_model.population_best_response),
        "p(c=40)": probability_at(higher_imitation_model.speed_distribution, 40),
    },
    "more_diffuse": {
        "best_response": asdict(more_diffuse_model.population_best_response),
        "p(c=40)": probability_at(more_diffuse_model.speed_distribution, 40),
    },
}


{'higher_imitation': {'best_response': {'a': 15.0,
   'b': 45.0,
   'c': 40.0,
   'expected_speed': 0.660476,
   'expected_pnl': 199976.96818607426},
  'p(c=40)': 0.282585},
 'more_diffuse': {'best_response': {'a': 15.0,
   'b': 45.0,
   'c': 40.0,
   'expected_speed': 0.6463640000000002,
   'expected_pnl': 194635.85817595758},
  'p(c=40)': 0.233285}}

In [21]:
checks = {
    "type_shares_sum_to_1": abs(sum(baseline_config.type_shares().values()) - 1.0) < 1e-12,
    "best_response_is_15_45_40": (
        baseline_model.population_best_response.a == 15.0
        and baseline_model.population_best_response.b == 45.0
        and baseline_model.population_best_response.c == 40.0
    ),
    "40_emerges_endogenously": probability_at(
        baseline_model.speed_distribution,
        40,
    ) > max(
        probability_at(baseline_model.speed_distribution, 35),
        probability_at(baseline_model.speed_distribution, 45),
    ),
    "higher_imitation_keeps_same_best": (
        higher_imitation_model.population_best_response.a == 15.0
        and higher_imitation_model.population_best_response.b == 45.0
        and higher_imitation_model.population_best_response.c == 40.0
    ),
    "more_diffuse_keeps_same_best": (
        more_diffuse_model.population_best_response.a == 15.0
        and more_diffuse_model.population_best_response.b == 45.0
        and more_diffuse_model.population_best_response.c == 40.0
    ),
}

checks


{'type_shares_sum_to_1': True,
 'best_response_is_15_45_40': True,
 '40_emerges_endogenously': True,
 'higher_imitation_keeps_same_best': True,
 'more_diffuse_keeps_same_best': True}